# 06 · Charts (Plotly)

Five interactive charts. They show here and are saved as standalone HTML to the `gold.export` volume
(download them into `docs/charts/` for GitHub Pages).

In [ ]:
%pip install -q "plotly>=5.24"

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))  # the repo's src/, via the Databricks Git folder

from favorita_spark.config import Config

cfg = Config()
spark.sql(f"USE CATALOG {cfg.catalog}")

In [ ]:
from favorita_spark import charts
from favorita_spark.stockout import weekly_flags

summary = spark.table("gold.promo_family_summary").toPandas()
events = spark.table("gold.fact_promo_event").select("family", "uplift", "touches_payday_or_holiday").toPandas()
weekly = weekly_flags(spark.table("gold.fact_stockout_run")).toPandas()
disp = spark.table("gold.item_dispersion").select("dispersion").toPandas()

figs = {
    "uplift_by_family": charts.uplift_by_family(summary),
    "uplift_vs_dip": charts.uplift_vs_dip(summary),
    "payday_check": charts.payday_check(events, cfg.min_family_events),
    "stockout_heatmap": charts.stockout_heatmap(weekly),
    "poisson_check": charts.poisson_check(disp),
}
for fig in figs.values():
    fig.show()

In [ ]:
charts.save_all(figs, f"/Volumes/{cfg.catalog}/gold/export/charts")